In [1]:
import os,sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname('utils'), '..')))
from module.utils import * 
from module.prompt import * 
from module.custom_model import *
from module.base_model import *
from module.tools import *

In [2]:
start_langsmith('development_1')

LangSmith 추적을 시작합니다.
[프로젝트명]
development_1


In [3]:
from typing import TypedDict, Annotated, List, Literal,Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage,ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks


In [4]:
class State(TypedDict):
    question : Annotated[str,'user input question']   # 사용자 질의 or requeustion 질의
    plan : Annotated[list[str],'get plan_node']  # llm 생성한 작업 계획서
    # messages : Annotated[list,add_messages]      # 작업 수행 후 얻은 데이터
    past_steps :Annotated[list, add_messages]
    answer : Annotated[str,' output final answer'] # 최종 답변 출력

In [5]:
def get_pdf_retriever_tool():
    loader = get_pdf_loader()
    splitter = get_text_splitter()
    docs = get_docs(loader,splitter) 
    embedding = get_embedding()
    retriever = get_retriever(docs,embedding)
    return get_retriever_tool(retriever)

def get_agent():
    prompt = get_prompt_agent()
    # weather_search_tool = get_weather()
    artist_news_search_tool = get_tavily_tool()
    # pdf_search_tool = get_pdf_retriever_tool()
    return create_react_agent(get_gpt(),[get_weather,artist_news_search_tool],prompt=prompt) 


In [6]:

def plan_node(state:State) -> State:
    llm = get_gemini()
    prompt = get_prompt_music_planner()
    chain = prompt | llm.with_structured_output(MusicPlan)
    reponse = chain.invoke({'messages':[state['question']]})
    return State({'plan':reponse.steps})

def execute_agent_node(state:State):
    agent = get_agent()
    plan = state['plan']
    plan_str = "\n".join(f"{i+1}. {text}" for i, text in enumerate(plan))
    task = plan[0]
    task_str = f"""For the following plan: \n\n {plan_str}\n\n You are tasked with executing [step 1. {task}]."""
    agent_response = agent.invoke({'messages':[task_str]})
    return State({'past_steps':[f" Question :{task}\n Response :{agent_response['messages'][-1].content}"]})

def decision_node(state:State):
    prompt = get_prompt_replanner()
    llm = get_gpt().with_structured_output(MusicAct)
    chain = prompt | llm
    outputs =  chain.invoke({'input':state['question'],'plan':state['plan'],'past_steps':state['past_steps']})
    # outputs => action=Plan(steps=['RAG의 작동 방식을 설명합니다.', 'RAG의 장단점을 설명합니다.', 'RAG의 활용 사례를 설명합니다.'])
    if isinstance(outputs.action,MusicResponse):
        return State({'answer':outputs.action.response})
    else:  # result == Plan 
        next_plan = outputs.action.steps
        if len(next_plan) == 0:
            return {"answer": "No more steps needed."}
        else:
            return {"plan": next_plan}

def final_generate_node(state: State):
    final_report = get_prompt_generate_markdown() | get_gemini() | StrOutputParser()
    answer = final_report.invoke({"input": state["question"], "past_steps": state['past_steps']})
    return {"answer": answer}

In [7]:
def should_continue(state:State)->Literal['final_generate_node','execute_agent_node']:
    if "answer" in state and state["answer"]:
        return 'final_generate_node'
    else:
        return 'execute_agent_node'

In [8]:
state_graph = StateGraph(State)
state_graph.add_node('plan_node',plan_node)
state_graph.add_node('execute_agent_node',execute_agent_node)
state_graph.add_node('decision_node',decision_node)
state_graph.add_node('final_generate_node',final_generate_node)


state_graph.add_edge(START,'plan_node')
state_graph.add_edge('plan_node','execute_agent_node')
state_graph.add_edge('execute_agent_node','decision_node')
state_graph.add_conditional_edges(
    source='decision_node',
    path=should_continue
)

state_graph.add_edge('final_generate_node',END)

ck = get_check_pointer()
graph = state_graph.compile(checkpointer=ck)

In [9]:
# visualize_graph(graph)
# print(graph.get_graph().draw_mermaid())

In [10]:
config = get_runnable_config(recursion_limit=10,thread_id=get_random_uuid())

inputs = {'question':'오늘 날씨에 듣기 좋은 노래 플레이리스트 작성해줘'}
invoke_graph(graph,inputs,config)


🔄 Node: plan_node 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
오늘 날씨에 어울리는 노래 20곡을 선정합니다.
선정된 노래들이 서비스 데이터베이스에 존재하는지 확인합니다.
최종 플레이리스트를 생성합니다.

🔄 Node: agent in [execute_agent_node] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_oeBKKB2NtzDcDiSXvudcy6ci)
 Call ID: call_oeBKKB2NtzDcDiSXvudcy6ci
  Args:
    location: 서울

🔄 Node: tools in [execute_agent_node] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================= Tool Message =================================
Name: get_weather

Seoul,KR의 현재 날씨: overcast clouds, 온도: 19.63°C

🔄 Node: agent in [execute_agent_node] 🔄
- - - - - - - - - - - - - - - - - - - - - - - - - 
================================== Ai Message ==================================

서울의 현재 날씨는 흐린 구름이 낀 상태이고, 온도는 약 19.6도입니다. 이 날씨에 어울리는 노래 20곡을 선정해드리겠습니다.

1. 아이유 - 밤편지
2. 백예린 - Square (2017)
3. 볼빨간사춘기 - 나만, 봄
4. 태연 - 사계
5. 

In [11]:
snapshot = graph.get_state(config)
snapshot.values

{'question': '오늘 날씨에 듣기 좋은 노래 플레이리스트 작성해줘',
 'plan': ['오늘 날씨에 어울리는 노래 20곡을 선정합니다.',
  '선정된 노래들이 서비스 데이터베이스에 존재하는지 확인합니다.',
  '최종 플레이리스트를 생성합니다.'],
 'past_steps': [HumanMessage(content=' Question :오늘 날씨에 어울리는 노래 20곡을 선정합니다.\n Response :오늘 서울의 날씨는 흐린 하늘에 기온은 약 20.6도입니다. 이 날씨에 어울리는 노래 20곡을 선정해드리겠습니다.\n\n1. 아이유 - 밤편지\n2. 박효신 - 야생화\n3. 백예린 - 그건 아마 우리의 잘못은 아닐 거야\n4. 적재 - 별 보러 가자\n5. 헤이즈 - 비도 오고 그래서\n6. 자이언티 - 꺼내 먹어요\n7. 볼빨간사춘기 - 나만, 봄\n8. 크러쉬 - 가끔\n9. 샘김 - 향수\n10. 정승환 - 이 바보야\n11. 폴킴 - 모든 날, 모든 순간\n12. 윤하 - 비밀번호 486\n13. 김광석 - 바람이 불어오는 곳\n14. 장범준 - 흔들리는 꽃들 속에서 네 샴푸향이 느껴진거야\n15. 10cm - 스토커\n16. 혁오 - 톡톡\n17. 선우정아 - 봄처녀\n18. 이소라 - 바람이 분다\n19. 자우림 - 매직 카펫 라이드\n20. 옥상달빛 - 수고했어, 오늘도\n\n이 노래들은 흐린 날씨와 어울리는 감성적이고 편안한 분위기의 곡들입니다. 다음 단계로 선정된 노래들이 서비스 데이터베이스에 존재하는지 확인해드릴까요?', additional_kwargs={}, response_metadata={}, id='87e39ebb-f008-4e55-8f3e-2277cd2e5499')],
 'answer': '## 오늘 날씨에 어울리는 플레이리스트 작성 보고서\n\n**작성일:** 2023년 10월 27일\n\n**작성자:** AI 어시스턴트\n\n### 1. 개요\n\n본 보고서는 오늘(2023년 10월 27일) 서울의 날씨인 흐린 하늘

In [12]:
from langgraph.prebuilt import create_react_agent
from langchain.chat_models import ChatOpenAI

# get_weather('서울')

# LLM 초기화
llm = get_gpt()

# 도구 리스트에 get_weather 포함
tools = [get_weather]

# 에이전트 생성
agent = create_react_agent(model=llm, tools=tools, prompt="You are a helpful assistant")

# 입력 예시
inputs = {"messages": [{"role": "user", "content": "서울,KR 날씨 "}]}

# 에이전트 실행
response = agent.invoke(inputs)
print(response)


{'messages': [HumanMessage(content='서울,KR 날씨 ', additional_kwargs={}, response_metadata={}, id='1412049c-bc41-4ea3-8e5e-07e379f61a59'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_BwN6KPrPL1M0wK7GHgTNBl66', 'function': {'arguments': '{"location":"서울,KR"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 58, 'total_tokens': 74, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245', 'id': 'chatcmpl-CMDQy4iEs9EKOgUYspmnpmsJBhEkN', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--4b019e91-acb6-4dcf-8dcc-7b55f359c3f7-0', tool_calls=[{'name': 'get_weather', 'args': {'location': '서울,KR'}, 'id': 'call_BwN6KPrPL1M0w